In [ ]:
# IPython magig  tools
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

from aind_vr_foraging_analysis.utils.parsing import data_access, parse
import aind_vr_foraging_analysis.utils.plotting as plotting
import aind_vr_foraging_analysis.utils as processing


# Plotting libraries
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages

import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime
import pytz

sns.set_context('talk')
from pathlib import Path
import warnings
pd.options.mode.chained_assignment = None  # Ignore SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import Rectangle

color1='#d95f02'
color2='#1b9e77'
color3='#7570b3'
color4='yellow'
odor_list_color = [color1, color2, color3, color4]

pdf_path = r'Z:\scratch\vr-foraging\sessions'
foraging_figures = r'C:\Users\tiffany.ona\OneDrive - Allen Institute\Documents'

from scipy.optimize import curve_fit

color_dict_label = {'InterSite': '#808080',
    'InterPatch': '#b3b3b3', 
    'PatchZ': '#d95f02', 'PatchZB': '#d95f02', 
    'PatchZA': '#d95f02', 
    'PatchB': '#d95f02','PatchA': '#7570b3', 
    'PatchC': '#1b9e77',
    'Alpha-pinene': '#1b9e77', 
    'Methyl Butyrate': '#7570b3', 
    'Amyl Acetate': '#d95f02', 
    'Fenchone': '#7570b3', 
    'S': color1,
    'D': color2,
    'N': color3,
    "odor_90": color1,
    "odor_60": color2,
    "odor_0": color3,
    'A': color1,
    'B': color2,
    'C': color3,
    'OdorA': color1,
    'OdorB': color2,
    'OdorC': color3,
    'odor_slow': color1,
    'odor_fast': color2,
    'A': color1,
    'B1': color2,
    'B2': color2,
    'C1': color3,
    'C2': color4
    }

label_dict = {**{
"InterSite": '#808080',
"InterPatch": '#b3b3b3'}, 
            **color_dict_label}

from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d


In [ ]:
pdf_path = r'Z:\scratch\vr-foraging\sessions'
base_path = r'Z:/scratch/vr-foraging/data/'
results_path = r'C:\Users\tiffany.ona\OneDrive - Allen Institute\Documents\VR foraging\manuscript\results\figures'
data_path = r'C:\git\Aind.Behavior.VrForaging.Analysis\data'

# Recover and clean batch 4 dataset
# batch4 = pd.read_csv(data_path + 'batch_4.csv') # if you want the original dataset
batch4 = pd.read_csv(os.path.join(data_path, 'batch_4_fixed_interpatch.csv'))

# These mice are in the dataset but didn't perform the manipulation
batch4 = batch4[(batch4['mouse'] != 754573)&(batch4['mouse'] != 754572)&(batch4['mouse'] != 745300)&(batch4['mouse'] != 745306)&(batch4['mouse'] != 745307)]

batch4["session"] = batch4["session"].apply(lambda x: str(x).split('_')[-1])
# batch4 = batch4[batch4['label'] == 'OdorSite']

## Micr with weird behavior
batch4 = batch4.loc[(batch4.mouse != 754577)&(batch4.mouse != 754575)]

batch4['shaping'] = np.where(batch4['experiment'].isin([
        'shaping_stageA_distanceA_stopA_v1',
        'shaping_stageA_distanceC_stopB_v1',
       'shaping_stageA_distanceD_stopC_v1',
       'shaping_stageA_distanceD_stopD_v1',
       'shaping_stageB_distanceD_stopD_v1',
       'shaping_stageB_distanceD_stopE_v1',
       'shaping_stageA_distanceB_stopB_v1',
       'shaping_stageA_distanceB_stopA_v1',
       'shaping_stageA_distanceD_stopB_v1',
       'shaping_stageA_distanceC_stopA_v1',
       'shaping_stageA_distanceC_stopC_v1',
       'stageC_v1'
       ]), True, False)

# df = df.loc[df.shaping == True]
batch4['stage'] = batch4['experiment'].str.extract(r'stage([A-Z])')[0]
batch4['stage'] = batch4['stage'].fillna('Graduation')

# # Import data from batch3
batch3 = pd.read_csv(os.path.join(data_path,  'batch_3_shaping.csv'))
batch3 = batch3.loc[(batch3.mouse != 694569)]

# Merge both datasets
df = pd.concat([batch3, batch4], ignore_index=True)

df= df.loc[~df.patch_label.isin(['patch_delayed', 'patch_no_reward', 'patch_single', 'delayed', 'single', 'no_reward', 'PatchZB'])]

df['patch_label'] = df['patch_label'].replace({'Alpha pinene': '60','Alpha-pinene': '60', 'Methyl Butyrate': '90', 'Ethyl Butyrate': '90', 'Amyl Acetate': '0', 
                                               '2,3-Butanedione': 'slow', '2-Heptanone': 'slow',  'Methyl Acetate':'fast', 'Fenchone':'0'})
df['experiment'] = df['experiment'].replace({'base': 'control'})

df['speed'] = df.length / df.duration_epoch

df.odor_sites = df.odor_sites.fillna(method='bfill')
df['last_intersite'] = df.last_site.shift(1)
df = df[df.last_intersite != 1]
df = df.loc[(df.label != "InterPatch")]

In [ ]:
import matplotlib.lines as mlines

df['is_choice'] = np.where(df.speed < 8, True, False)

mask = (df.session_n == 4) & (df.mouse == 754570)
palette = {'InterSite': "#ABABAB", "OdorSite": '#d95f02'}
fig, ax = plt.subplots(figsize=(6,4))

sns.scatterplot(data=df.loc[mask & (df.is_choice == 1)], x='odor_sites', y='speed', hue='label', palette = palette, marker='.', edgecolor=None,s=100)
for label, color in palette.items():
    subset = df.loc[mask & (df.label == label)& (df.is_choice == 0)]
    plt.scatter(subset['odor_sites'], subset['speed'],
                facecolors='none', edgecolors=color,
                marker='.', s=100, linewidths=1.5, alpha=0.8)
plt.ylim(-2,40)
plt.xlabel('Number of Odor Sites')
plt.ylabel('Speed (cm/s)')

filled_handles = [
    mlines.Line2D([], [], marker='o', linestyle='None',
                  markerfacecolor=color, markeredgecolor='none',
                  markersize=8, label=label)
    for label, color in palette.items()
]

choice_handles = [
    mlines.Line2D([], [], marker='o', linestyle='None',
                  markerfacecolor='black', markeredgecolor='black',
                  markersize=8, label='Stop'),
    mlines.Line2D([], [], marker='o', linestyle='None',
                  markerfacecolor='none', markeredgecolor='black',
                  markersize=8, markeredgewidth=1.5, label='Non-stop'),
]

plt.legend(
    handles=filled_handles + choice_handles,
    title='Epoch Type',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)
sns.despine()

In [ ]:
mouse = 745302
start_session = 1

In [ ]:

sessions = [start_session, start_session + 1, start_session + 2]
palette = {'InterSite': "#ABABAB", "OdorSite": "#fcbf19ea"}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, session in zip(axes, sessions):
    mask = (df.session_n == session) & (df.mouse == mouse)

    # Filled — stop (is_choice == 1)
    sns.scatterplot(data=df.loc[mask & (df.is_choice == 1)], 
                    x='odor_sites', y='speed', hue='label',
                    palette=palette, marker='.', edgecolor=None, 
                    s=100, ax=ax, legend=False)

    # Hollow — non-stop (is_choice == 0)
    for label, color in palette.items():
        subset = df.loc[mask & (df.label == label) & (df.is_choice == 0)]
        ax.scatter(subset['odor_sites'], subset['speed'],
                   facecolors='none', edgecolors=color,
                   marker='.', s=100, linewidths=1.5, alpha=0.8)

    ax.set_title(f'Session {session}')
    ax.set_xlabel('Number of Odor Sites')
    ax.set_ylim(-2, 40)
    # ax.set_xlim(-0.5, 200)
    sns.despine(ax=ax)


axes[0].set_ylabel('Speed (cm/s)')
for ax in axes[1:]:
    ax.set_ylabel('')

# Single shared legend on the last axis
filled_handles = [
    mlines.Line2D([], [], marker='o', linestyle='None',
                  markerfacecolor=color, markeredgecolor='none',
                  markersize=8, label=label)
    for label, color in palette.items()
]
choice_handles = [
    mlines.Line2D([], [], marker='o', linestyle='None',
                  markerfacecolor='black', markeredgecolor='black',
                  markersize=8, label='Stop'),
    mlines.Line2D([], [], marker='o', linestyle='None',
                  markerfacecolor='none', markeredgecolor='black',
                  markersize=8, markeredgewidth=1.5, label='Non-stop'),
]

axes[-1].legend(
    handles=filled_handles + choice_handles,
    title='Epoch Type',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()

In [ ]:
pre_df = df.loc[df['stage'] == 'B']
plot = pre_df.loc[pre_df.engaged == 1].groupby(['mouse', 'session_n', 'site_number'])['is_choice'].mean().reset_index()
plot.site_number -=1
plot['leave'] = 1 - plot['is_choice']

In [ ]:
plot_filtered = plot.loc[(plot.site_number > 0)].copy()
plot_filtered['leave'] = plot_filtered['leave'].astype(float)
plot_filtered['session_rel'] = plot_filtered.groupby('mouse')['session_n'].transform(
    lambda x: x - x.min() + 1
)

cum_df = (
    plot_filtered
    .sort_values('site_number')
    .groupby(['mouse', 'session_rel', 'site_number'])['leave']  # ← add session_rel
    .mean()
    .groupby(level=['mouse', 'session_rel'])                     # ← cumsum within each session
    .cumsum()
    .reset_index()
)

cum_df['leave_cum'] = cum_df.groupby(['mouse', 'session_rel'])['leave'].transform(
    lambda x: x / x.max()                                        # ← normalize per mouse+session
)

cum_df = cum_df.loc[cum_df.session_rel < 4] 

# Find the last site where at least N mice are present per session
min_mice = 4  # ← adjust this threshold

mouse_counts_site = cum_df.groupby(['session_rel', 'site_number'])['mouse'].nunique().reset_index()
mouse_counts_site.columns = ['session_rel', 'site_number', 'n_mice']

# Last valid site per session
last_valid_site = (
    mouse_counts_site.loc[mouse_counts_site.n_mice >= min_mice]
    .groupby('session_rel')['site_number']
    .max()
    .reset_index()
)
last_valid_site.columns = ['session_rel', 'last_valid_site']

# Truncate each session at its last valid site
cum_df_filtered = cum_df.merge(last_valid_site, on='session_rel')
cum_df = cum_df_filtered.loc[cum_df_filtered.site_number <= cum_df_filtered.last_valid_site]

fig, ax = plt.subplots(figsize=(3.5,3))
sns.lineplot(data=cum_df, x='site_number', y='leave_cum',
             hue='session_rel', palette=['coral', 'crimson', 'darkred'], errorbar='sd', err_kws={'linewidth': 0})

sites = np.arange(1, 11)

# Fit exponential: f(x) = a * exp(-b * x)
# f(1) = 0.9, f(10) = 0.7
a = 0.9
b = -np.log(0.7 / 0.9) / (10 - 1)
curve = a * np.exp(-b * (sites - 1))

plt.plot(sites, curve, color='k', label='Decay curve', linestyle='--', linewidth=1)
plt.axvline(10, ymin=0, ymax=0.7, color='k', linestyle='--', linewidth=1)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Session')
plt.xlabel('Number of stops')
plt.xlim(0.5, 20)
plt.ylim(0, 1)
plt.yticks(np.arange(0, 1.1, 0.5))
plt.ylabel('Cumulative \n probability of leaving')
sns.despine()

In [ ]:
plot_filtered = plot.loc[plot.site_number > 0].copy()
plot_filtered['leave'] = plot_filtered['leave'].astype(float)
plot_filtered['session_rel'] = plot_filtered.groupby('mouse')['session_n'].transform(
    lambda x: x - x.min() + 1  # starts at 1
)

plot_filtered = plot_filtered.loc[plot_filtered.session_rel == 3]

cum_df = (
    plot_filtered
    .sort_values('site_number')
    .groupby(['mouse', 'site_number'])['leave']
    .mean()
    .groupby(level='mouse')
    .cumsum()
    .reset_index()
)

cum_df['leave_cum'] = cum_df.groupby('mouse')['leave'].transform(lambda x: x / x.max())

sns.lineplot(data=cum_df, x='site_number', y='leave_cum',
             hue='mouse', palette='tab20', errorbar=None)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Mouse')
plt.xlabel('Site Number')
plt.xlim(0.5, 20)
plt.ylim(0, 1)
plt.ylabel('Cumulative probability of leaving')
sns.despine()

In [ ]:
# Find patches where mouse stopped at least once
valid_patches = (
    pre_df.groupby(['mouse', 'session_n', 'patch_number'])['is_choice']
    .max()  # 1 if any row has is_choice == 1
    .reset_index()
)
valid_patches = valid_patches.loc[valid_patches.is_choice == 1, ['mouse', 'session_n', 'patch_number']]
valid_patches['is_valid_patch'] = True

pre_df = pre_df.merge(valid_patches, on=['mouse', 'session_n', 'patch_number'], how='left')
pre_df['is_valid_patch'] = pre_df['is_valid_patch'].fillna(False)

# Only increment counter when patch changes AND the new patch is valid
patch_change = pre_df['patch_number'].ne(pre_df['patch_number'].shift())
valid_patch_change = patch_change & pre_df['is_valid_patch']

pre_df['cumulative_patch_count'] = (
    valid_patch_change.groupby([pre_df['mouse'], pre_df['session_n']])
    .cumsum()
)

pre_df['norm_start_time'] = pre_df.groupby(['mouse', 'session_n'])['start_time'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)
pre_df['round_norm_start_time'] = pre_df['norm_start_time'].round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(3*2, 3))
palette = ['coral', 'crimson', 'darkred']


group = pre_df.loc[~pre_df.mouse.isin(['754575', '754582'])].groupby(['mouse','session', 'stage', 'session_n', 'round_norm_start_time']).cumulative_patch_count.mean().reset_index()
group['session_rel'] = group.groupby('mouse')['session_n'].transform(
    lambda x: x - x.min() + 1
)

sns.lineplot(data=group.loc[(group.session_rel < 4)], x='round_norm_start_time', marker='o',  y='cumulative_patch_count', palette=palette, hue='session_rel', ci=None, ax=ax, legend='brief', linewidth=2.5, alpha=0.8)
sns.despine()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Session')
plt.ylim(0, 30)
ax.set_xlabel('Normalized time')
ax.set_ylabel('Cumulative Patch Count')
fig.savefig(os.path.join(results_path, f'cumulative_patch_count_shaping_all_mice.pdf'), dpi=300, bbox_inches='tight')

In [ ]:
# Define corrections
corrections = [
    # Single session
    {"mouse": 754559, "sessions": [7,8], "stage": "C"},
    {"mouse": 754559, "condition": lambda s: s > 8, "stage": "Graduation"},
    {"mouse": 754560, "condition": lambda s: s > 9, "stage": "Graduation"},
    {"mouse": 754579, "sessions": [6,7], "stage": "C"},
    {"mouse": 754566, "sessions": [4,5], "stage": "B"},
    {"mouse": 754579, "condition": lambda s: s > 7, "stage": "Graduation"},
    {"mouse": 754570, "condition": lambda s: s > 12, "stage": "Graduation"},
    {"mouse": 754574, "condition": lambda s: s > 11, "stage": "Graduation"},
    {"mouse": 754582, "sessions": [9,10], "stage": "C"},
    {"mouse": 754582, "condition": lambda s: s > 10, "stage": "Graduation"},
    {"mouse": 754580, "sessions": [9,10], "stage": "C"},
    {"mouse": 754580, "condition": lambda s: s > 10, "stage": "Graduation"},
]

# Apply corrections
for corr in corrections:
    mask = (df['mouse'] == corr["mouse"])
    if "sessions" in corr:
        mask &= df['session_n'].isin(corr["sessions"])
    elif "condition" in corr:
        mask &= corr["condition"](df['session_n'])
    
    df.loc[mask, 'stage'] = corr["stage"]

In [ ]:
import matplotlib.patches as mpatches

# Map stages to ordered ranks
stage_order = sorted(df['stage'].unique())
stage_to_rank = {s: i for i, s in enumerate(stage_order)}
df['stage_rank'] = df['stage'].map(stage_to_rank)

# One row per mouse/session
session_stage = (
    df.groupby(['mouse', 'session_n'])['stage']
    .first()
    .reset_index()
)
session_stage = session_stage.loc[session_stage.session_n >1]
session_stage['session_n'] -= 1

session_stage['stage_rank'] = session_stage['stage'].map(stage_to_rank)

# Color palette per stage
n_stages = len(stage_order)
stage_palette = dict(zip(stage_order, sns.color_palette('tab10', n_stages)))

fig, ax = plt.subplots(1, 1, figsize=(6,8))

mice = session_stage['mouse'].unique()
mouse_to_y = {m: i for i, m in enumerate(mice)}

for _, row in session_stage.iterrows():
    ax.barh(y=mouse_to_y[row['mouse']], 
            width=1, 
            left=row['session_n'] - 1,
            color=stage_palette[row['stage']], 
            edgecolor='white', linewidth=0.5)

ax.set_yticks(range(len(mice)))
ax.set_yticklabels(mice)
ax.set_xlabel('Relative Session')
ax.set_ylabel('Mouse')
ax.set_title('Stage progression (Gantt)')
handles = [mpatches.Patch(color=stage_palette[s], label=s.replace('shaping_', ''))
           for s in stage_order]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
ax.set_xlim(0, 15)
sns.despine(ax=ax)

plt.tight_layout()

In [ ]:
# Map stages to ordered ranks
stage_order = ['A','B','C','Graduation']  # explicitly ordered
stage_to_rank = {s: i for i, s in enumerate(stage_order)}
df['stage_rank'] = df['stage'].map(stage_to_rank)

# One row per mouse/session
session_stage = (
    df.groupby(['mouse', 'session_n'])['stage']
    .first()
    .reset_index()
)

session_stage = session_stage.loc[session_stage.session_n > 1]
session_stage['session_n'] -= 1
session_stage['stage_rank'] = session_stage['stage'].map(stage_to_rank)

# Add graduation stage for each mouse (one session after their last)
last_sessions = session_stage.groupby('mouse')['session_n'].max().reset_index()
last_sessions['session_n'] = last_sessions['session_n'] + 1
last_sessions['stage'] = 'Graduation'
session_stage_plot = pd.concat([session_stage, last_sessions], ignore_index=True)

# Sequential color palette for progression
# n_stages = len(stage_order)

palette_div = sns.diverging_palette(220, 20, n=n_stages)
# stage_palette = dict(zip(stage_order, palette_div))

# Explicit stage order
stage_order = ['A', 'B', 'C', 'Graduation']

# Custom colors
stage_palette = {
    'A': "#E0E0E0",          # blue
    'B': "#989898",          # red
    'C': "#717171",          # green
    'Graduation': "k"  # black
}
# Map mice to y-axis
fig, ax = plt.subplots(1, 1, figsize=(5, 7))
mice = session_stage_plot['mouse'].unique()
mouse_to_y = {m: i for i, m in enumerate(mice)}

# Remap to consecutive session index per mouse (no gaps)
session_stage_plot = session_stage_plot.sort_values(['mouse', 'session_n'])
session_stage_plot['session_rel'] = session_stage_plot.groupby('mouse').cumcount() + 1

# Plot Gantt-like bars
for _, row in session_stage_plot.iterrows():
    ax.barh(
        y=mouse_to_y[row['mouse']],
        width=1,
        left=row['session_rel'] - 1,
        color=stage_palette[row['stage']],
        edgecolor='white',
        linewidth=0.5
    )

# Labels
ax.set_yticks(range(len(mice)))
ax.set_yticklabels(mice)
ax.set_xlabel('Session')
ax.set_ylabel('Mouse')
ax.set_title('Stage progression (Gantt)')

# Legend
handles = [mpatches.Patch(color=stage_palette[s], label=s) for s in stage_order]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)

ax.set_xlim(0, session_stage_plot['session_rel'].max() + 1)
sns.despine(ax=ax)
plt.xlim(0, 15)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'stage_progression_gantt_all_mice.pdf'), dpi=300, bbox_inches='tight')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

order = ['A','B','C','Graduation']
stage_map = {k:i for i,k in enumerate(order)}

df['stage_num'] = df['stage'].map(stage_map)

fig, ax = plt.subplots(figsize=(4,4))

mice = df['mouse'].unique()
offsets = np.linspace(-0.15, 0.15, len(mice))
mouse_offset = dict(zip(mice, offsets))

# grayscale palette: dark to light
grays = np.linspace(0.2, 0.8, len(mice))

# individual mice
for (mouse, g), gray in zip(df.sort_values('session_n').groupby('mouse'), grays):
    ax.step(
        g['session_n'],
        g['stage_num'],
        where='post',
        color=str(gray),
        alpha=0.2
    )

# mean progression
mean_stage = df.groupby('session_n')['stage_num'].mean().sort_index()

ax.plot(
    mean_stage.index,
    mean_stage.values,
    linewidth=3,
    color='k',
)

ax.set_yticks(range(len(order)))
ax.set_yticklabels(order)
ax.set_xlim(0, 18)

ax.set_xlabel("Session")
ax.set_ylabel("Stage")
sns.despine()

## SAME BUT FOR NEW BATCHES